# Step 1: Fractal Estimator Calibration
Brain / Universe / AI trifold project.

Goal: learn what our dimension estimators can and cannot distinguish BEFORE touching real data.
Every test set below has a **known label** (true dimension, or "not a fractal").

Rules fixed in advance (no cherry-picking after seeing results):
- Same estimators and same scale windows for every dataset.
- Scale windows are defined as fractions of each dataset's extent.
- A dataset only gets called "scale-invariant" if both estimators agree AND the slope is stable across the window.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree

SEED = 42
rng = np.random.default_rng(SEED)

# Pre-registered settings
N_POINTS = 8000
CORR_WINDOW = (0.02, 0.15)   # fraction of extent, correlation dimension
BOX_WINDOW = (0.05, 0.30)    # fraction of extent, box counting
SLOPE_TOL = 0.35             # max allowed slope drift lower-half vs upper-half of window
AGREE_TOL = 0.35             # max allowed gap between the two estimators

## Labeled test generators

In [ ]:
def ifs(shifts, ratio, n, dim):
    """Chaos-game IFS: x -> ratio*x + shift. Similarity dimension = log(k)/log(1/ratio)."""
    shifts = np.asarray(shifts, float)
    x = rng.random(dim)
    out = np.empty((n, dim))
    ks = rng.integers(0, len(shifts), n + 50)
    for i, k in enumerate(ks):
        x = ratio * x + shifts[k]
        if i >= 50:
            out[i - 50] = x
    return out

def sierpinski_triangle(n):
    s = [(0, 0), (0.5, 0), (0.25, np.sqrt(3) / 4)]
    return ifs(s, 0.5, n, 2), np.log(3) / np.log(2)

def sierpinski_carpet(n):
    s = [(i / 3, j / 3) for i in range(3) for j in range(3) if not (i == 1 and j == 1)]
    return ifs(s, 1 / 3, n, 2) * (3 / 2), np.log(8) / np.log(3)

def menger_sponge(n):
    s = [(i / 3, j / 3, k / 3) for i in range(3) for j in range(3) for k in range(3)
         if (i == 1) + (j == 1) + (k == 1) < 2]
    return ifs(s, 1 / 3, n, 3) * (3 / 2), np.log(20) / np.log(3)

def soneira_peebles(eta=4, lam=1.8, levels=7):
    """Hierarchical clustering model built to mimic galaxy-like fractal structure. D = log(eta)/log(lam)."""
    pts = np.zeros((1, 3))
    r = 1.0
    for _ in range(levels):
        r /= lam
        parents = np.repeat(pts, eta, axis=0)
        d = rng.normal(size=parents.shape)
        d /= np.linalg.norm(d, axis=1, keepdims=True)
        d *= r * rng.random((len(parents), 1)) ** (1 / 3)
        pts = parents + d
    return pts, np.log(eta) / np.log(lam)

def uniform_cube(n):
    return rng.random((n, 3)), 3.0

def uniform_square_in_3d(n):
    p = np.zeros((n, 3)); p[:, :2] = rng.random((n, 2)); return p, 2.0

def line_in_3d(n):
    p = np.zeros((n, 3)); p[:, 0] = rng.random(n); return p, 1.0

def brownian_path_3d(n):
    return np.cumsum(rng.normal(size=(n, 3)), axis=0), 2.0

def clustered_not_fractal(n, n_parents=40, sigma=0.02):
    """Neyman-Scott clusters: lumpy, but with ONE characteristic scale. NOT a fractal. False-positive test."""
    parents = rng.random((n_parents, 3))
    idx = rng.integers(0, n_parents, n)
    return parents[idx] + rng.normal(scale=sigma, size=(n, 3)), None

## Estimators (identical for every dataset)

In [ ]:
def normalize(pts):
    pts = pts - pts.min(axis=0)
    return pts / pts.max()

def corr_integral(pts, radii):
    tree = cKDTree(pts)
    n = len(pts)
    c = tree.count_neighbors(tree, radii)
    return (c - n) / (n * (n - 1))

def slope(logx, logy):
    return np.polyfit(logx, logy, 1)[0]

def corr_dim(pts, window=CORR_WINDOW, n_r=14):
    pts = normalize(pts)
    radii = np.logspace(np.log10(window[0]), np.log10(window[1]), n_r)
    c = corr_integral(pts, radii)
    ok = c > 0
    lr, lc = np.log(radii[ok]), np.log(c[ok])
    half = len(lr) // 2
    return slope(lr, lc), slope(lr[:half], lc[:half]), slope(lr[half:], lc[half:]), radii, c

def box_dim(pts, window=BOX_WINDOW, n_s=10):
    pts = normalize(pts)
    eps = np.logspace(np.log10(window[0]), np.log10(window[1]), n_s)
    counts = np.array([len(np.unique(np.floor(pts / e), axis=0)) for e in eps])
    return -slope(np.log(eps), np.log(counts))

def subsample_std(pts, frac=0.8, reps=15):
    est = []
    for _ in range(reps):
        sub = pts[rng.choice(len(pts), int(frac * len(pts)), replace=False)]
        est.append(corr_dim(sub)[0])
    return np.std(est)

## Run the calibration battery

In [ ]:
tests = {
    "Line (D=1)": line_in_3d(N_POINTS),
    "Sierpinski triangle": sierpinski_triangle(N_POINTS),
    "Brownian path 3D (D=2)": brownian_path_3d(N_POINTS),
    "Square in 3D (D=2)": uniform_square_in_3d(N_POINTS),
    "Sierpinski carpet": sierpinski_carpet(N_POINTS),
    "Menger sponge": menger_sponge(N_POINTS),
    "Soneira-Peebles (galaxy-like)": soneira_peebles(),
    "Uniform cube / Poisson null (D=3)": uniform_cube(N_POINTS),
    "Clustered, NOT fractal": clustered_not_fractal(N_POINTS),
}

rows, curves = [], {}
for name, (pts, true_d) in tests.items():
    d_corr, d_lo, d_hi, radii, c = corr_dim(pts)
    d_box = box_dim(pts)
    drift = d_hi - d_lo
    stable = abs(drift) < SLOPE_TOL
    agree = abs(d_corr - d_box) < AGREE_TOL
    verdict = "scale-invariant" if (stable and agree) else "scale-dependent / unreliable"
    rows.append({
        "dataset": name, "N": len(pts),
        "true_D": None if true_d is None else round(true_d, 3),
        "corr_D": round(d_corr, 3), "box_D": round(d_box, 3),
        "slope_drift": round(drift, 3),
        "corr_std": round(subsample_std(pts), 3),
        "verdict": verdict,
    })
    curves[name] = (radii, c)

df = pd.DataFrame(rows)
df["corr_error"] = (df["corr_D"] - df["true_D"]).round(3)
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 20)
print(df.to_string(index=False))
df.to_csv("calibration_results.csv", index=False)

## Correlation integrals
A true fractal gives a straight line on log-log axes across the window. A clustered-but-not-fractal set bends.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for name, (radii, c) in curves.items():
    ax.loglog(radii, c, marker="o", ms=3, label=name)
ax.axvspan(*CORR_WINDOW, color="grey", alpha=0.08)
ax.set_xlabel("r (fraction of extent)"); ax.set_ylabel("C(r)")
ax.legend(fontsize=7); ax.set_title("Correlation integrals, calibration battery")
plt.tight_layout(); plt.savefig("calibration_curves.png", dpi=150); plt.show()

## How to read this
- `corr_error` is the estimator's bias on cases where we know the answer. **Any real-data result is only trustworthy to within roughly this error.**
- The clustered control is the key test: if it comes back "scale-invariant," our method produces false fractals and we must tighten it before moving on.
- Next: apply this exact code, unchanged, to one real dataset (TNG halos, neuron morphologies, or model activations).